### Setup

In [ ]:
import json
import os
import re
import time
from collections import Counter
from datetime import datetime

import main

In [ ]:
exec_datetime = datetime.now().strftime("%Y%m%d%H%M%S")
print(exec_datetime)

In [ ]:
headers = {
    "User-Agent": "EsportUsernames/1.0 (https://github.com/hollowscene)"
}

In [ ]:
main_wiki_list = [
    "dota2",
    "counterstrike",
    "mobilelegends",
    "apexlegends",
    "overwatch",
    "pubgmobile",
    "smash",
    "honorofkings",
    "warcraft",
    "starcraft",
    "easportsfc",
    "hearthstone",
    "valorant",
    "rocketleague",
    "leagueoflegends",
    "rainbowsix",
    "starcraft2",
    "ageofempires",
    "fighters",
    "pubg",
    "brawlstars",
    "wildrift",
    "heroes",
    "artifact",
]

## Import

In [ ]:
def ingest_and_clean(wiki, pattern=re.compile(r".+\ \(.+\)$"), data_subfolder="data"):

    players = main.get_player_pages(wiki=wiki, headers=headers)
    player_names = [player["title"] for player in players if not player["title"].startswith("Category:")]

    # Cache raw names
    with open(os.path.join(data_subfolder, f"{wiki}_player_names_raw.txt"), "w") as f_raw:
        f_raw.write(json.dumps(player_names))

    cleaned_player_names = [main.clean_username(pattern, player_name) for player_name in player_names]
    # value_counts = Counter([player_name for player_name in cleaned_player_names])

    # print(f"Total player name count in {wiki} wiki: {len(player_names):,}")
    # print(f"Unique player name count in {wiki} wiki: {len(value_counts):,}")

    # Cache cleaned names
    with open(os.path.join(data_subfolder, f"{wiki}_player_names_cleaned.txt"), "w") as f_cleaned:
        f_cleaned.write(json.dumps(cleaned_player_names))

In [ ]:
data_subfolder = os.path.join("data", exec_datetime)
if not os.path.exists(data_subfolder):
    os.makedirs(data_subfolder)

In [ ]:
for wiki in main_wiki_list:
    start_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{start_datetime} - {wiki} import and clean start")
    ingest_and_clean(wiki, data_subfolder=data_subfolder)
    end_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{end_datetime} - {wiki} import and clean finished")
    time.sleep(0.5)

## Analysis

In [ ]:
# Re-use cached imported data
exec_datetime = "20241206223049"
data_subfolder = os.path.join("data", exec_datetime)

In [ ]:
player_names_by_wiki = {}

In [ ]:
for wiki in main_wiki_list:
    # with open(os.path.join(data_subfolder, f"{wiki}_player_names_cleaned.txt")) as f_cleaned:
    #     cleaned_player_names = json.load(f_cleaned)
    # value_counts = Counter([player_name for player_name in cleaned_player_names])
    # player_names_by_wiki[wiki] = value_counts

    # Transform so player name is key
    with open(os.path.join(data_subfolder, f"{wiki}_player_names_cleaned.txt")) as f_cleaned:

        cleaned_player_names = json.load(f_cleaned)

        for player_name in cleaned_player_names:

            if player_name not in player_names_by_wiki:
                player_names_by_wiki[player_name] = {}
                player_names_by_wiki[player_name]["total"] = 0

            if wiki not in player_names_by_wiki[player_name]:
                player_names_by_wiki[player_name][wiki] = 0

            player_names_by_wiki[player_name]["total"] += 1
            player_names_by_wiki[player_name][wiki] += 1

In [ ]:
player_names_by_wiki

In [ ]:
len(player_names_by_wiki)

In [ ]:
player_names_by_wiki["Rich"]

Note: Rich has a page on both the League of Legends and Heroes of the Storm wikis. Consolidation may require parsing of the page content.